# ポストアポカリプス・ソーラーパンク探索ゲーム
# Post-Apocalyptic Solar-Punk Roguelike Explorer

荒廃した世界で緑の技術を探索するローグライクゲーム

A roguelike exploration game set in a post-apocalyptic world where nature and solar technology are reclaiming the wasteland.

## 操作方法 / Controls:
- **w/a/s/d**: 移動 (Move)
- **i**: インベントリ表示 (Show inventory)
- **p**: アイテムを拾う (Pick up item)
- **q**: ゲーム終了 (Quit game)

In [ ]:
import random
import time
from enum import Enum
from dataclasses import dataclass
from typing import List, Tuple, Optional
from IPython.display import clear_output, display, HTML
import sys

In [ ]:
# ========================================
# ゲームの定数とEnums / Game Constants and Enums
# ========================================

class Tile(Enum):
    """マップタイル / Map Tiles"""
    FLOOR = '.'
    WALL = '#'
    PLAYER = '@'
    ENEMY = 'E'
    ITEM = '*'
    SOLAR_PANEL = 'S'
    PLANT = '♣'
    RUINS = '%'
    RADIATION = '~'
    WATER = '≈'
    TREE = '♠'
    EXIT = '>'

class ItemType(Enum):
    """アイテムタイプ / Item Types"""
    SOLAR_BATTERY = "ソーラーバッテリー"
    HEALING_HERB = "薬草"
    CLEAN_WATER = "浄水"
    TECH_SCRAP = "テクノスクラップ"
    SEED = "植物の種"
    RADIATION_SUIT = "放射線防護服"

class EnemyType(Enum):
    """敵タイプ / Enemy Types"""
    MUTANT_RAT = "変異ネズミ"
    BROKEN_ROBOT = "故障ロボット"
    WILD_DOG = "野犬"
    RADIATION_BEAST = "放射線獣"

# カラーコード / Color codes for terminal display
COLORS = {
    'player': '#00FF00',      # Green
    'enemy': '#FF0000',       # Red
    'item': '#FFFF00',        # Yellow
    'solar': '#FFA500',       # Orange
    'plant': '#90EE90',       # Light Green
    'water': '#00BFFF',       # Deep Sky Blue
    'radiation': '#8B008B',   # Dark Magenta
    'wall': '#808080',        # Gray
    'floor': '#D3D3D3',       # Light Gray
}

In [ ]:
# ========================================
# データクラス / Data Classes
# ========================================

@dataclass
class Position:
    """位置情報 / Position information"""
    x: int
    y: int
    
    def __add__(self, other):
        return Position(self.x + other.x, self.y + other.y)
    
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

@dataclass
class Item:
    """アイテム / Item"""
    name: str
    item_type: ItemType
    value: int
    description: str

@dataclass
class Enemy:
    """敵キャラクター / Enemy Character"""
    name: str
    enemy_type: EnemyType
    hp: int
    attack: int
    position: Position
    
@dataclass
class Player:
    """プレイヤーキャラクター / Player Character"""
    name: str
    hp: int
    max_hp: int
    attack: int
    defense: int
    position: Position
    inventory: List[Item]
    level: int
    exp: int

In [ ]:
# ========================================
# プロシージャルマップ生成 / Procedural Map Generation
# ========================================

class MapGenerator:
    """プロシージャルマップジェネレーター / Procedural Map Generator"""
    
    def __init__(self, width: int = 60, height: int = 25):
        self.width = width
        self.height = height
        self.map = [[Tile.WALL for _ in range(width)] for _ in range(height)]
        self.rooms = []
        
    def generate(self) -> List[List[Tile]]:
        """マップを生成 / Generate map"""
        # ランダムに部屋を生成
        num_rooms = random.randint(6, 10)
        
        for _ in range(num_rooms):
            width = random.randint(4, 10)
            height = random.randint(4, 8)
            x = random.randint(1, self.width - width - 1)
            y = random.randint(1, self.height - height - 1)
            
            new_room = (x, y, width, height)
            
            # 他の部屋と重ならないかチェック
            if not any(self._rooms_overlap(new_room, other) for other in self.rooms):
                self._create_room(new_room)
                
                # 前の部屋と繋ぐ
                if self.rooms:
                    prev_room = self.rooms[-1]
                    self._create_corridor(new_room, prev_room)
                
                self.rooms.append(new_room)
        
        # ソーラーパンク要素を追加
        self._add_solar_elements()
        # 植物を追加
        self._add_vegetation()
        # 廃墟を追加
        self._add_ruins()
        # 放射線ゾーンを追加
        self._add_radiation_zones()
        # 水源を追加
        self._add_water_sources()
        # 出口を追加
        self._add_exit()
        
        return self.map
    
    def _rooms_overlap(self, room1, room2) -> bool:
        """部屋が重なるかチェック / Check if rooms overlap"""
        x1, y1, w1, h1 = room1
        x2, y2, w2, h2 = room2
        return (x1 <= x2 + w2 and x1 + w1 >= x2 and
                y1 <= y2 + h2 and y1 + h1 >= y2)
    
    def _create_room(self, room):
        """部屋を作成 / Create a room"""
        x, y, width, height = room
        for i in range(y, y + height):
            for j in range(x, x + width):
                if 0 <= i < self.height and 0 <= j < self.width:
                    self.map[i][j] = Tile.FLOOR
    
    def _create_corridor(self, room1, room2):
        """部屋を繋ぐ廊下を作成 / Create corridor between rooms"""
        x1, y1, w1, h1 = room1
        x2, y2, w2, h2 = room2
        
        # 部屋の中心点
        cx1, cy1 = x1 + w1 // 2, y1 + h1 // 2
        cx2, cy2 = x2 + w2 // 2, y2 + h2 // 2
        
        # L字型の廊下を作成
        if random.random() < 0.5:
            # 横→縦
            for x in range(min(cx1, cx2), max(cx1, cx2) + 1):
                if 0 <= cy1 < self.height and 0 <= x < self.width:
                    self.map[cy1][x] = Tile.FLOOR
            for y in range(min(cy1, cy2), max(cy1, cy2) + 1):
                if 0 <= y < self.height and 0 <= cx2 < self.width:
                    self.map[y][cx2] = Tile.FLOOR
        else:
            # 縦→横
            for y in range(min(cy1, cy2), max(cy1, cy2) + 1):
                if 0 <= y < self.height and 0 <= cx1 < self.width:
                    self.map[y][cx1] = Tile.FLOOR
            for x in range(min(cx1, cx2), max(cx1, cx2) + 1):
                if 0 <= cy2 < self.height and 0 <= x < self.width:
                    self.map[cy2][x] = Tile.FLOOR
    
    def _add_solar_elements(self):
        """ソーラーパネルを追加 / Add solar panels"""
        for _ in range(random.randint(3, 6)):
            pos = self._get_random_floor_position()
            if pos:
                self.map[pos.y][pos.x] = Tile.SOLAR_PANEL
    
    def _add_vegetation(self):
        """植物を追加 / Add vegetation"""
        for _ in range(random.randint(10, 20)):
            pos = self._get_random_floor_position()
            if pos:
                tile = Tile.PLANT if random.random() < 0.7 else Tile.TREE
                self.map[pos.y][pos.x] = tile
    
    def _add_ruins(self):
        """廃墟を追加 / Add ruins"""
        for _ in range(random.randint(5, 10)):
            pos = self._get_random_floor_position()
            if pos:
                self.map[pos.y][pos.x] = Tile.RUINS
    
    def _add_radiation_zones(self):
        """放射線ゾーンを追加 / Add radiation zones"""
        for _ in range(random.randint(2, 4)):
            pos = self._get_random_floor_position()
            if pos:
                # 小さな放射線エリアを作成
                for dy in range(-1, 2):
                    for dx in range(-1, 2):
                        ny, nx = pos.y + dy, pos.x + dx
                        if (0 <= ny < self.height and 0 <= nx < self.width and 
                            self.map[ny][nx] == Tile.FLOOR and random.random() < 0.6):
                            self.map[ny][nx] = Tile.RADIATION
    
    def _add_water_sources(self):
        """水源を追加 / Add water sources"""
        for _ in range(random.randint(2, 4)):
            pos = self._get_random_floor_position()
            if pos:
                # 小さな水たまりを作成
                for dy in range(-1, 2):
                    for dx in range(-1, 2):
                        ny, nx = pos.y + dy, pos.x + dx
                        if (0 <= ny < self.height and 0 <= nx < self.width and 
                            self.map[ny][nx] == Tile.FLOOR and random.random() < 0.5):
                            self.map[ny][nx] = Tile.WATER
    
    def _add_exit(self):
        """出口を追加 / Add exit"""
        if self.rooms:
            last_room = self.rooms[-1]
            x, y, w, h = last_room
            exit_x = x + w // 2
            exit_y = y + h // 2
            if 0 <= exit_y < self.height and 0 <= exit_x < self.width:
                self.map[exit_y][exit_x] = Tile.EXIT
    
    def _get_random_floor_position(self) -> Optional[Position]:
        """ランダムな床の位置を取得 / Get random floor position"""
        attempts = 0
        while attempts < 100:
            x = random.randint(1, self.width - 2)
            y = random.randint(1, self.height - 2)
            if self.map[y][x] == Tile.FLOOR:
                return Position(x, y)
            attempts += 1
        return None
    
    def get_spawn_position(self) -> Position:
        """プレイヤーのスポーン位置を取得 / Get player spawn position"""
        if self.rooms:
            x, y, w, h = self.rooms[0]
            return Position(x + w // 2, y + h // 2)
        return Position(1, 1)

In [ ]:
# ========================================
# ゲームエンジン / Game Engine
# ========================================

class Game:
    """ゲームメインクラス / Main Game Class"""
    
    def __init__(self):
        self.map_generator = MapGenerator()
        self.game_map = self.map_generator.generate()
        self.width = len(self.game_map[0])
        self.height = len(self.game_map)
        
        # プレイヤー初期化
        spawn_pos = self.map_generator.get_spawn_position()
        self.player = Player(
            name="サバイバー",
            hp=100,
            max_hp=100,
            attack=10,
            defense=5,
            position=spawn_pos,
            inventory=[],
            level=1,
            exp=0
        )
        
        # 敵を配置
        self.enemies = self._spawn_enemies()
        
        # アイテムを配置
        self.items = self._spawn_items()
        
        self.messages = []
        self.game_over = False
        self.turn = 0
        
    def _spawn_enemies(self) -> List[Enemy]:
        """敵を生成 / Spawn enemies"""
        enemies = []
        enemy_types = [
            (EnemyType.MUTANT_RAT, 20, 5),
            (EnemyType.WILD_DOG, 30, 8),
            (EnemyType.BROKEN_ROBOT, 40, 12),
            (EnemyType.RADIATION_BEAST, 50, 15),
        ]
        
        for _ in range(random.randint(5, 10)):
            pos = self.map_generator._get_random_floor_position()
            if pos and pos != self.player.position:
                enemy_type, hp, attack = random.choice(enemy_types)
                enemy = Enemy(
                    name=enemy_type.value,
                    enemy_type=enemy_type,
                    hp=hp,
                    attack=attack,
                    position=pos
                )
                enemies.append(enemy)
        
        return enemies
    
    def _spawn_items(self) -> dict:
        """アイテムを生成 / Spawn items"""
        items = {}
        item_types = [
            (ItemType.SOLAR_BATTERY, "エネルギーを蓄える", 20),
            (ItemType.HEALING_HERB, "体力を20回復", 20),
            (ItemType.CLEAN_WATER, "体力を10回復", 10),
            (ItemType.TECH_SCRAP, "技術の欠片", 15),
            (ItemType.SEED, "新しい命の種", 10),
            (ItemType.RADIATION_SUIT, "放射線から保護", 30),
        ]
        
        for _ in range(random.randint(8, 15)):
            pos = self.map_generator._get_random_floor_position()
            if pos:
                item_type, desc, value = random.choice(item_types)
                item = Item(
                    name=item_type.value,
                    item_type=item_type,
                    value=value,
                    description=desc
                )
                items[(pos.x, pos.y)] = item
        
        return items
    
    def move_player(self, dx: int, dy: int):
        """プレイヤーを移動 / Move player"""
        new_pos = Position(self.player.position.x + dx, self.player.position.y + dy)
        
        # 境界チェック
        if not (0 <= new_pos.x < self.width and 0 <= new_pos.y < self.height):
            return
        
        # 壁チェック
        tile = self.game_map[new_pos.y][new_pos.x]
        if tile == Tile.WALL:
            self.add_message("壁があって進めない")
            return
        
        # 敵との衝突チェック
        enemy = self._get_enemy_at(new_pos)
        if enemy:
            self._combat(enemy)
            return
        
        # 移動実行
        self.player.position = new_pos
        
        # 特殊タイルの効果
        self._check_tile_effect(tile)
        
        # 出口に到達
        if tile == Tile.EXIT:
            self._next_level()
        
        # ターン進行
        self.turn += 1
        self._enemy_turn()
    
    def _check_tile_effect(self, tile: Tile):
        """タイルの効果をチェック / Check tile effects"""
        if tile == Tile.RADIATION:
            damage = random.randint(3, 8)
            self.player.hp -= damage
            self.add_message(f"放射線でダメージ！ -{damage} HP")
            if self.player.hp <= 0:
                self._game_over()
        elif tile == Tile.SOLAR_PANEL:
            heal = random.randint(5, 10)
            self.player.hp = min(self.player.max_hp, self.player.hp + heal)
            self.add_message(f"ソーラーパネルからエネルギーを得た！ +{heal} HP")
        elif tile == Tile.WATER:
            heal = 5
            self.player.hp = min(self.player.max_hp, self.player.hp + heal)
            self.add_message(f"きれいな水を見つけた！ +{heal} HP")
        elif tile == Tile.PLANT or tile == Tile.TREE:
            if random.random() < 0.3:
                heal = 3
                self.player.hp = min(self.player.max_hp, self.player.hp + heal)
                self.add_message(f"植物から養分を得た！ +{heal} HP")
    
    def _combat(self, enemy: Enemy):
        """戦闘処理 / Combat processing"""
        # プレイヤーの攻撃
        damage_to_enemy = max(1, self.player.attack - random.randint(0, 3))
        enemy.hp -= damage_to_enemy
        self.add_message(f"{enemy.name}に{damage_to_enemy}ダメージを与えた！")
        
        if enemy.hp <= 0:
            self.enemies.remove(enemy)
            exp_gain = enemy.attack * 2
            self.player.exp += exp_gain
            self.add_message(f"{enemy.name}を倒した！ +{exp_gain} EXP")
            self._check_level_up()
            return
        
        # 敵の反撃
        damage_to_player = max(1, enemy.attack - self.player.defense)
        self.player.hp -= damage_to_player
        self.add_message(f"{enemy.name}から{damage_to_player}ダメージを受けた！")
        
        if self.player.hp <= 0:
            self._game_over()
    
    def _enemy_turn(self):
        """敵のターン / Enemy turn"""
        for enemy in self.enemies[:]:
            # プレイヤーが近くにいるかチェック
            dist = abs(enemy.position.x - self.player.position.x) + abs(enemy.position.y - self.player.position.y)
            
            if dist <= 5:
                # プレイヤーに近づく
                dx = 0 if enemy.position.x == self.player.position.x else (1 if enemy.position.x < self.player.position.x else -1)
                dy = 0 if enemy.position.y == self.player.position.y else (1 if enemy.position.y < self.player.position.y else -1)
                
                new_pos = Position(enemy.position.x + dx, enemy.position.y + dy)
                
                # 移動可能かチェック
                if (0 <= new_pos.x < self.width and 0 <= new_pos.y < self.height and
                    self.game_map[new_pos.y][new_pos.x] != Tile.WALL and
                    not self._get_enemy_at(new_pos)):
                    
                    # プレイヤーと同じ位置なら攻撃
                    if new_pos == self.player.position:
                        damage = max(1, enemy.attack - self.player.defense)
                        self.player.hp -= damage
                        self.add_message(f"{enemy.name}から{damage}ダメージ！")
                        if self.player.hp <= 0:
                            self._game_over()
                    else:
                        enemy.position = new_pos
    
    def _get_enemy_at(self, pos: Position) -> Optional[Enemy]:
        """指定位置の敵を取得 / Get enemy at position"""
        for enemy in self.enemies:
            if enemy.position == pos:
                return enemy
        return None
    
    def pick_up_item(self):
        """アイテムを拾う / Pick up item"""
        pos_key = (self.player.position.x, self.player.position.y)
        if pos_key in self.items:
            item = self.items.pop(pos_key)
            self.player.inventory.append(item)
            self.add_message(f"{item.name}を拾った！")
            
            # アイテムの即座効果
            if item.item_type == ItemType.HEALING_HERB:
                self.player.hp = min(self.player.max_hp, self.player.hp + 20)
                self.add_message("体力が回復した！ +20 HP")
            elif item.item_type == ItemType.CLEAN_WATER:
                self.player.hp = min(self.player.max_hp, self.player.hp + 10)
                self.add_message("体力が回復した！ +10 HP")
        else:
            self.add_message("ここにはアイテムがない")
    
    def _check_level_up(self):
        """レベルアップチェック / Check level up"""
        exp_needed = self.player.level * 50
        if self.player.exp >= exp_needed:
            self.player.level += 1
            self.player.exp -= exp_needed
            self.player.max_hp += 20
            self.player.hp = self.player.max_hp
            self.player.attack += 3
            self.player.defense += 2
            self.add_message(f"レベルアップ！ Lv.{self.player.level}")
    
    def _next_level(self):
        """次のレベルへ / Go to next level"""
        self.add_message("次のエリアへ進む...")
        
        # 新しいマップを生成
        self.map_generator = MapGenerator()
        self.game_map = self.map_generator.generate()
        
        # プレイヤー位置をリセット
        self.player.position = self.map_generator.get_spawn_position()
        
        # 敵とアイテムを再配置
        self.enemies = self._spawn_enemies()
        self.items = self._spawn_items()
        
        self.add_message("新しいエリアに到着した")
    
    def _game_over(self):
        """ゲームオーバー / Game over"""
        self.game_over = True
        self.add_message("ゲームオーバー...")
    
    def add_message(self, msg: str):
        """メッセージを追加 / Add message"""
        self.messages.append(msg)
        if len(self.messages) > 5:
            self.messages.pop(0)
    
    def render(self) -> str:
        """ゲーム画面を描画 / Render game screen"""
        output = []
        
        # タイトル
        output.append("=" * 80)
        output.append("ポストアポカリプス・ソーラーパンク探索ゲーム")
        output.append("=" * 80)
        output.append("")
        
        # マップ描画
        for y in range(self.height):
            line = ""
            for x in range(self.width):
                pos = Position(x, y)
                
                # プレイヤー位置
                if pos == self.player.position:
                    line += Tile.PLAYER.value
                # 敵の位置
                elif self._get_enemy_at(pos):
                    line += Tile.ENEMY.value
                # アイテムの位置
                elif (x, y) in self.items:
                    line += Tile.ITEM.value
                # マップタイル
                else:
                    line += self.game_map[y][x].value
            
            output.append(line)
        
        output.append("")
        
        # ステータス表示
        output.append("=" * 80)
        output.append(f"プレイヤー: {self.player.name} | Lv.{self.player.level} | HP: {self.player.hp}/{self.player.max_hp} | "
                     f"攻撃: {self.player.attack} | 防御: {self.player.defense} | EXP: {self.player.exp}")
        output.append(f"ターン: {self.turn} | アイテム: {len(self.player.inventory)}個 | 敵: {len(self.enemies)}体")
        output.append("=" * 80)
        
        # メッセージ
        output.append("\n【メッセージ】")
        for msg in self.messages:
            output.append(f"  {msg}")
        
        output.append("\n【凡例】")
        output.append(f"  @ = あなた | E = 敵 | * = アイテム | # = 壁 | . = 床")
        output.append(f"  S = ソーラーパネル | ♣ = 植物 | ♠ = 樹木 | ~ = 放射線")
        output.append(f"  ≈ = 水 | % = 廃墟 | > = 出口")
        
        output.append("\n【操作】w:上 a:左 s:下 d:右 | p:アイテムを拾う | i:インベントリ | q:終了")
        
        return "\n".join(output)
    
    def show_inventory(self) -> str:
        """インベントリを表示 / Show inventory"""
        output = ["\n【インベントリ】"]
        if not self.player.inventory:
            output.append("  (空っぽ)")
        else:
            for i, item in enumerate(self.player.inventory, 1):
                output.append(f"  {i}. {item.name} - {item.description}")
        return "\n".join(output)

In [ ]:
# ========================================
# ゲームループ / Game Loop
# ========================================

def play_game():
    """ゲームを開始 / Start the game"""
    print("ゲームを初期化中...")
    game = Game()
    game.add_message("ポストアポカリプスの世界へようこそ")
    game.add_message("ソーラー技術と自然を探索して生き延びよ")
    
    while not game.game_over:
        # 画面をクリアして再描画
        clear_output(wait=True)
        print(game.render())
        
        # 入力を待つ
        try:
            command = input("\nコマンドを入力: ").lower().strip()
            
            if command == 'w':
                game.move_player(0, -1)
            elif command == 'a':
                game.move_player(-1, 0)
            elif command == 's':
                game.move_player(0, 1)
            elif command == 'd':
                game.move_player(1, 0)
            elif command == 'p':
                game.pick_up_item()
            elif command == 'i':
                print(game.show_inventory())
                input("\nEnterキーで続ける...")
            elif command == 'q':
                print("\nゲームを終了します...")
                break
            else:
                game.add_message("無効なコマンド")
        
        except KeyboardInterrupt:
            print("\n\nゲームを中断しました")
            break
        except Exception as e:
            print(f"\nエラーが発生しました: {e}")
            break
    
    # ゲーム終了
    clear_output(wait=True)
    print(game.render())
    print("\n" + "=" * 80)
    if game.game_over:
        print("ゲームオーバー！")
        print(f"最終スコア - レベル: {game.player.level}, ターン: {game.turn}, 倒した敵: {game.turn // 2}")
    else:
        print("またお会いしましょう！")
    print("=" * 80)

## ゲームを開始 / Start Game

下のセルを実行してゲームを開始します！

Run the cell below to start the game!

In [ ]:
# ゲームを開始 / Start the game
play_game()

## ゲームの特徴 / Game Features

### ポストアポカリプス要素 / Post-Apocalyptic Elements:
- 廃墟 (%) - 過去の文明の痕跡
- 放射線ゾーン (~) - 危険な汚染地帯
- 変異生物との戦闘

### ソーラーパンク要素 / Solar-Punk Elements:
- ソーラーパネル (S) - クリーンエネルギーで回復
- 植物と樹木 (♣♠) - 自然の再生
- きれいな水源 (≈) - 生命の源

### ローグライク要素 / Roguelike Elements:
- プロシージャル生成マップ
- ターン制戦闘
- レベルアップシステム
- アイテム収集
- パーマデス（死んだらゲームオーバー）

### 探索要素 / Exploration Elements:
- 複数の部屋とエリア
- 出口を見つけて次のレベルへ
- アイテムと敵の配置
- 環境との相互作用